# ClearBank – Análise Financeira de Transações

**Analista:** Desenvolvedor Junior  
**Empresa:** ClearBank Fintech  
**Python:** 3.10+

## Como executar

1. Certifique-se de que o arquivo `transacoes.csv` está na **mesma pasta** que este notebook.
2. Execute as células **em ordem** de cima para baixo (Kernel → Restart & Run All).
3. A célula de execução principal chamará todas as funções e gerará:
   - Relatório formatado no terminal (saída da célula)
   - Arquivo `relatorio.json` na mesma pasta
   - Arquivo `grafico.png` com o gráfico mensal

## Estrutura do `transacoes.csv`

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| id | inteiro | Identificador único da transação |
| data | texto | Data no formato AAAA-MM-DD |
| cliente_id | texto | Código do cliente (não pode ser vazio) |
| tipo | texto | `credito` ou `debito` |
| valor | decimal | Valor da transação (deve ser maior que 0) |
| descricao | texto | Descrição livre da operação |
| categoria | texto | Ex.: salario, compra, transferencia |

In [ ]:
import csv
import json
from datetime import datetime

# ── Constantes ────────────────────────────────────────────────────────────────
LIMITE_SUSPEITO: float = 10_000.00
ARQUIVO_CSV: str = "transacoes.csv"
ARQUIVO_JSON: str = "relatorio.json"
ARQUIVO_GRAFICO: str = "grafico.png"
TIPOS_VALIDOS: set[str] = {"credito", "debito"}

print("Constantes carregadas:")
print(f"  LIMITE_SUSPEITO = R$ {LIMITE_SUSPEITO:,.2f}")
print(f"  ARQUIVO_CSV     = {ARQUIVO_CSV}")
print(f"  ARQUIVO_JSON    = {ARQUIVO_JSON}")

In [ ]:
def ler_transacoes(filepath: str) -> list[dict]:
    """Lê o arquivo CSV e retorna a lista bruta de transações como dicionários."""
    try:
        with open(filepath, encoding="utf-8", newline="") as arquivo:
            leitor = csv.DictReader(arquivo)
            return list(leitor)
    except FileNotFoundError:
        print(f"ERRO: Arquivo '{filepath}' não encontrado.")
        print("Crie o arquivo transacoes.csv antes de executar o notebook.")
        return []


# ── Teste rápido ──────────────────────────────────────────────────────────────
linhas_brutas = ler_transacoes(ARQUIVO_CSV)
print(f"Linhas lidas do CSV: {len(linhas_brutas)}")
print("Primeira linha:", linhas_brutas[0] if linhas_brutas else "(nenhuma)")

In [ ]:
def validar_transacao(row: dict) -> dict | None:
    """
    Valida e converte uma linha bruta do CSV.
    Retorna um dict com tipos corretos se válida, ou None se inválida.
    """
    # Validar e converter id
    try:
        id_val = int(row.get("id", "").strip())
    except (ValueError, AttributeError):
        return None

    # Validar cliente_id
    cliente_id = row.get("cliente_id", "").strip()
    if not cliente_id:
        return None

    # Validar e converter data
    try:
        data = datetime.strptime(row.get("data", "").strip(), "%Y-%m-%d")
    except (ValueError, AttributeError):
        return None

    # Validar tipo
    tipo = row.get("tipo", "").strip().lower()
    if tipo not in TIPOS_VALIDOS:
        return None

    # Validar e converter valor
    try:
        valor = float(row.get("valor", "").strip())
        if valor <= 0:
            return None
    except (ValueError, AttributeError):
        return None

    return {
        "id": id_val,
        "data": data,
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": row.get("descricao", "").strip(),
        "categoria": row.get("categoria", "").strip(),
    }


# ── Teste rápido ──────────────────────────────────────────────────────────────
linha_valida = {"id": "1", "data": "2026-01-05", "cliente_id": "CLI001",
                "tipo": "credito", "valor": "3500.00", "descricao": "Salário", "categoria": "salario"}
linha_invalida = {"id": "abc", "data": "2026-01-05", "cliente_id": "CLI001",
                  "tipo": "credito", "valor": "100.00", "descricao": "Teste", "categoria": "compra"}

resultado_valido = validar_transacao(linha_valida)
resultado_invalido = validar_transacao(linha_invalida)

print("Linha válida   →", resultado_valido)
print("Linha inválida →", resultado_invalido)

In [ ]:
def processar_transacoes(linhas_brutas: list[dict]) -> tuple[list[dict], int]:
    """
    Itera sobre as linhas brutas, valida cada uma e retorna
    a lista de transações válidas e a contagem de inválidas.
    Imprime o resumo da limpeza no terminal.
    """
    validas: list[dict] = []
    total = len(linhas_brutas)

    for row in linhas_brutas:
        transacao = validar_transacao(row)
        if transacao is not None:
            validas.append(transacao)

    invalidas = total - len(validas)

    print("\n===== RESUMO DA LIMPEZA =====")
    print(f"Total de linhas lidas: {total}")
    print(f"Linhas válidas:        {len(validas)}")
    print(f"Linhas inválidas:      {invalidas}")

    return validas, invalidas


# ── Teste rápido ──────────────────────────────────────────────────────────────
transacoes_validas, qtd_invalidas = processar_transacoes(linhas_brutas)
print(f"\nTransações válidas prontas para análise: {len(transacoes_validas)}")

In [ ]:
def calcular_periodo(transacoes: list[dict]) -> tuple[datetime, datetime, int]:
    """
    Retorna a data mais antiga, a mais recente e a quantidade de dias
    entre elas a partir de uma lista de transações válidas.
    """
    datas = [t["data"] for t in transacoes]
    data_mais_antiga = min(datas)
    data_mais_recente = max(datas)
    dias_periodo = (data_mais_recente - data_mais_antiga).days
    return data_mais_antiga, data_mais_recente, dias_periodo


def gerar_relatorio(transacoes: list[dict], qtd_invalidas: int) -> dict:
    """
    Agrupa as transações por mês e calcula métricas financeiras.
    Identifica transações suspeitas (valor > LIMITE_SUSPEITO).
    Retorna estrutura compatível com o JSON de saída.
    """
    resumo_mensal: dict[str, dict] = {}
    suspeitas: list[dict] = []

    for t in transacoes:
        mes = t["data"].strftime("%Y-%m")

        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "saldo": 0.0,
                "media": 0.0,
                "maior_valor": 0.0,
                "menor_valor": float("inf"),
                "_soma_valores": 0.0,
            }

        m = resumo_mensal[mes]
        m["quantidade"] += 1
        m["_soma_valores"] += t["valor"]

        if t["tipo"] == "credito":
            m["total_credito"] += t["valor"]
        else:
            m["total_debito"] += t["valor"]

        if t["valor"] > m["maior_valor"]:
            m["maior_valor"] = t["valor"]
        if t["valor"] < m["menor_valor"]:
            m["menor_valor"] = t["valor"]

        if t["valor"] > LIMITE_SUSPEITO:
            suspeitas.append({
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data"].strftime("%Y-%m-%d"),
                "valor": t["valor"],
            })

    # Pós-processamento: calcular saldo, média e limpar campo interno
    for mes, m in resumo_mensal.items():
        m["saldo"] = round(m["total_credito"] - m["total_debito"], 2)
        m["media"] = round(m["_soma_valores"] / m["quantidade"], 2)
        m["total_credito"] = round(m["total_credito"], 2)
        m["total_debito"] = round(m["total_debito"], 2)
        m["maior_valor"] = round(m["maior_valor"], 2)
        m["menor_valor"] = round(m["menor_valor"], 2)
        del m["_soma_valores"]

    data_antiga, data_recente, dias = calcular_periodo(transacoes)

    return {
        "gerado_em": datetime.today().strftime("%Y-%m-%d"),
        "total_transacoes_validas": len(transacoes),
        "total_transacoes_invalidas": qtd_invalidas,
        "periodo": {
            "inicio": data_antiga.strftime("%Y-%m-%d"),
            "fim": data_recente.strftime("%Y-%m-%d"),
            "dias": dias,
        },
        "resumo_mensal": dict(sorted(resumo_mensal.items())),
        "transacoes_suspeitas": suspeitas,
    }


# ── Teste rápido ──────────────────────────────────────────────────────────────
relatorio = gerar_relatorio(transacoes_validas, qtd_invalidas)
print("Meses encontrados:", list(relatorio["resumo_mensal"].keys()))
print("Transações suspeitas:", len(relatorio["transacoes_suspeitas"]))
print("Período:", relatorio["periodo"])

In [ ]:
def formatar_moeda(valor: float) -> str:
    """Formata um float no padrão monetário brasileiro: R$ 1.840,25"""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def exibir_relatorio(relatorio: dict) -> None:
    """Imprime o relatório mensal formatado no terminal."""
    periodo = relatorio["periodo"]

    print("\n" + "=" * 50)
    print("       RELATÓRIO FINANCEIRO – CLEARBANK")
    print("=" * 50)
    print(f"Gerado em:              {relatorio['gerado_em']}")
    print(f"Período analisado:      {periodo['inicio']} → {periodo['fim']}")
    print(f"Dias no período:        {periodo['dias']}")
    print(f"Transações válidas:     {relatorio['total_transacoes_validas']}")
    print(f"Transações inválidas:   {relatorio['total_transacoes_invalidas']}")

    print("\n===== RELATÓRIO MENSAL =====")
    for mes, dados in relatorio["resumo_mensal"].items():
        print(f"\nMês: {mes}")
        print(f"  Transações:    {dados['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(dados['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(dados['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(dados['saldo'])}")
        print(f"  Média:         {formatar_moeda(dados['media'])}")
        print(f"  Maior valor:   {formatar_moeda(dados['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(dados['menor_valor'])}")

    print("\n===== TRANSAÇÕES SUSPEITAS =====")
    suspeitas = relatorio["transacoes_suspeitas"]
    if suspeitas:
        for s in suspeitas:
            print(f"ID: {s['id']} | Cliente: {s['cliente_id']} | Data: {s['data']} | Valor: {formatar_moeda(s['valor'])}")
    else:
        print("Nenhuma transação suspeita encontrada.")

    print("\n" + "=" * 50)


# ── Teste rápido ──────────────────────────────────────────────────────────────
print("Teste formatação:", formatar_moeda(1840.25))
print("Teste formatação:", formatar_moeda(15000.00))

In [ ]:
def salvar_json(relatorio: dict, filepath: str) -> None:
    """Salva o relatório em formato JSON no caminho especificado."""
    try:
        with open(filepath, "w", encoding="utf-8") as arquivo:
            json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)
        print(f"Relatório JSON salvo em: {filepath}")
    except OSError as e:
        print(f"ERRO ao salvar JSON: {e}")


# ── Teste rápido ──────────────────────────────────────────────────────────────
import os
salvar_json(relatorio, ARQUIVO_JSON)
if os.path.exists(ARQUIVO_JSON):
    tamanho = os.path.getsize(ARQUIVO_JSON)
    print(f"Arquivo gerado com {tamanho} bytes.")

In [ ]:
# ── CÉLULA DE EXECUÇÃO PRINCIPAL ─────────────────────────────────────────────
# Execute esta célula para rodar o pipeline completo do zero.

print("Iniciando análise financeira ClearBank...")
print(f"Lendo arquivo: {ARQUIVO_CSV}\n")

# 1. Leitura
linhas_brutas = ler_transacoes(ARQUIVO_CSV)

if linhas_brutas:
    # 2. Validação e limpeza
    transacoes_validas, qtd_invalidas = processar_transacoes(linhas_brutas)

    if transacoes_validas:
        # 3. Geração do relatório
        relatorio = gerar_relatorio(transacoes_validas, qtd_invalidas)

        # 4. Exibição no terminal
        exibir_relatorio(relatorio)

        # 5. Exportação JSON
        salvar_json(relatorio, ARQUIVO_JSON)
    else:
        print("AVISO: Nenhuma transação válida encontrada. Verifique o CSV.")
else:
    print("AVISO: Nenhuma linha lida. Verifique se o arquivo CSV existe e está correto.")

---
## Requisito Opcional 2 – Visualização com matplotlib

Gráfico de barras empilhadas com crédito e débito por mês (Opção C do enunciado).

In [ ]:
import matplotlib.pyplot as plt

def gerar_grafico(relatorio: dict, filepath: str) -> None:
    """Gera gráfico de barras empilhadas (crédito e débito por mês) e salva como imagem."""
    meses = list(relatorio["resumo_mensal"].keys())
    creditos = [relatorio["resumo_mensal"][m]["total_credito"] for m in meses]
    debitos = [relatorio["resumo_mensal"][m]["total_debito"] for m in meses]

    x = range(len(meses))
    largura = 0.5

    fig, ax = plt.subplots(figsize=(10, 6))

    barras_credito = ax.bar(x, creditos, largura, label="Crédito", color="#2ecc71")
    barras_debito = ax.bar(x, debitos, largura, bottom=creditos, label="Débito", color="#e74c3c")

    ax.set_title("ClearBank – Crédito e Débito por Mês", fontsize=15, fontweight="bold", pad=15)
    ax.set_xlabel("Mês", fontsize=12)
    ax.set_ylabel("Valor (R$)", fontsize=12)
    ax.set_xticks(list(x))
    ax.set_xticklabels(meses, rotation=30, ha="right")
    ax.legend(fontsize=11)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"R$ {v:,.0f}"))
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    # Rótulos de valor nas barras de crédito
    for bar, val in zip(barras_credito, creditos):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
                    f"R$ {val:,.0f}", ha="center", va="center", fontsize=8, color="white", fontweight="bold")

    # Rótulos de valor nas barras de débito
    for bar, base, val in zip(barras_debito, creditos, debitos):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, base + val / 2,
                    f"R$ {val:,.0f}", ha="center", va="center", fontsize=8, color="white", fontweight="bold")

    plt.tight_layout()
    plt.savefig(filepath, dpi=150)
    plt.show()
    print(f"Gráfico salvo em: {filepath}")


gerar_grafico(relatorio, ARQUIVO_GRAFICO)

---
## Requisito Opcional 1 – Análise com pandas

A análise alternativa usando pandas está implementada no arquivo separado `analise_pandas.py`.  
Execute-o com: `python analise_pandas.py`  

Os resultados serão comparados automaticamente com os obtidos pelo método nativo acima.